# Quiet Sun Magnetic Fields: Implementation
# 조용한 태양 자기장: 구현

**Paper**: Bellot Rubio & Orozco Suárez (2019), *Living Reviews in Solar Physics* 16:1  
**DOI**: 10.1007/s41116-018-0017-1

## Overview / 개요

**English.** This notebook reproduces the key observational diagnostics of quiet-Sun magnetism described in the review. We build a synthetic magnetogram featuring a kG network and a hectogauss internetwork, compute Stokes V signals via the weak-field approximation at different filling factors, fit a lognormal PDF to the internetwork field-strength distribution, model Hanle depolarization as a function of field strength, investigate filling-factor retrieval, and study how the unsigned longitudinal flux scales with spatial resolution (reproducing the empirical $\phi \propto \exp(-1.1 r)$ law).

**한국어.** 이 노트북은 리뷰에 기술된 조용한 태양 자기장의 핵심 관측적 진단들을 재현한다. kG network와 hectogauss internetwork를 가진 합성 magnetogram을 구축하고, 약자장 근사로 다른 충전 인자에서 Stokes V 신호를 계산하며, internetwork 자기장 세기 분포에 lognormal PDF를 적합하고, 자기장 세기에 따른 Hanle 탈편광을 모델링하고, 충전 인자 검색을 조사하며, 부호 없는 종단 자속이 공간 분해능에 따라 어떻게 스케일되는지 연구한다(경험 법칙 $\phi \propto \exp(-1.1 r)$ 재현).

In [ ]:
"""Imports for quiet-Sun magnetic field diagnostics.

Uses NumPy for arrays, SciPy for stats and filtering, Matplotlib for plots.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, ndimage, optimize

np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. Synthetic Quiet-Sun Magnetogram / 합성 조용한 태양 Magnetogram

**English.** We construct a 200 x 200 pixel magnetogram simulating a supergranular cell. The **network** consists of a few kG flux concentrations (B ~ 1.2-1.5 kG, filling factor ~0.7) at cell boundaries. The **internetwork** is a sea of weak (~100-200 G), randomly oriented horizontal fields inside the cell. This reproduces the essential structure seen in Hinode/SP observations (Lites et al. 2008).

**한국어.** Supergranular cell을 시뮬레이션하는 200 x 200 픽셀 magnetogram을 구성한다. **Network**는 cell 경계의 몇 개 kG 자속 집중체(B ~ 1.2-1.5 kG, 충전 인자 ~0.7)로 구성된다. **Internetwork**는 cell 내부의 약한(~100-200 G), 무작위 방향의 수평 자기장 바다이다. 이는 Hinode/SP 관측(Lites et al. 2008)에서 보이는 핵심 구조를 재현한다.

In [ ]:
def build_synthetic_magnetogram(n=200, network_patches=12, grid_scale=30.0):
    """Construct a synthetic longitudinal magnetogram of a supergranular cell.

    Args:
        n: Linear grid size (n x n pixels).
        network_patches: Number of kG network concentrations at cell boundary.
        grid_scale: Characteristic scale of supergranule in pixels (cell radius).

    Returns:
        Tuple of arrays:
            B_field: intrinsic field strength map (G).
            incl: inclination map (deg).
            B_los: longitudinal field component = B*cos(incl) (G, signed).
    """
    x = np.arange(n)
    y = np.arange(n)
    xx, yy = np.meshgrid(x, y)

    # Internetwork: turbulent weak fields, lognormal distribution peaked at ~120 G
    rv = stats.lognorm(s=0.8, scale=120.0)
    B_in = rv.rvs(size=(n, n))
    B_in = np.clip(B_in, 10.0, 800.0)
    # Inclinations predominantly horizontal (90 deg) with scatter
    incl_in = 90.0 + 25.0 * np.random.randn(n, n)

    B = B_in.copy()
    incl = incl_in.copy()

    # Network: kG patches arranged near the boundary of the cell
    cx, cy = n / 2, n / 2
    for _ in range(network_patches):
        theta = np.random.uniform(0, 2 * np.pi)
        r = grid_scale + 5.0 * np.random.randn()
        px = cx + r * np.cos(theta)
        py = cy + r * np.sin(theta)
        sigma = 2.0 + np.random.rand()
        amp = np.random.uniform(1200.0, 1600.0)
        patch = amp * np.exp(-((xx - px) ** 2 + (yy - py) ** 2) / (2 * sigma ** 2))
        mask = patch > 50.0
        B = np.where(patch > B, patch, B)
        incl = np.where(mask, 10.0 + 15.0 * np.random.randn(), incl)
        if np.random.rand() < 0.5:
            incl[mask] = 180.0 - incl[mask]

    B_los = B * np.cos(np.deg2rad(incl))
    return B, incl, B_los


B_field, inclination, B_los = build_synthetic_magnetogram()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(B_field, cmap="inferno", origin="lower", vmin=0, vmax=1500)
axes[0].set_title("|B| intrinsic (G)\n내재적 자기장 세기")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(inclination, cmap="twilight", origin="lower", vmin=0, vmax=180)
axes[1].set_title("Inclination (deg)\n경사각")
plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(B_los, cmap="RdBu_r", origin="lower", vmin=-500, vmax=500)
axes[2].set_title("B_LOS = B cos(gamma) (G)\n종단 자기장")
plt.colorbar(im2, ax=axes[2])
for ax in axes:
    ax.set_xlabel("x [pixel]")
    ax.set_ylabel("y [pixel]")
plt.tight_layout()
plt.show()

## 2. Stokes V Signal at Different Filling Factors / 다른 충전 인자에서의 Stokes V 신호

**English.** The weak-field approximation gives
$$ V(\lambda) = -f \, \Delta\lambda_B \, \cos\gamma \, \frac{dI_0}{d\lambda}, $$
with $\Delta\lambda_B = 4.6686\times 10^{-10}\,\lambda_0^2 g_\text{eff} B$ (mÅ, Å, G). We synthesize the Fe I 6302.5 Å line ($g_\text{eff}=2.5$) as a Gaussian absorption profile, compute $dI_0/d\lambda$, and vary the filling factor $f$ from 0.05 to 0.8 at fixed $B = 300$ G, $\gamma = 30°$.

**한국어.** 약자장 근사는
$$ V(\lambda) = -f \, \Delta\lambda_B \, \cos\gamma \, \frac{dI_0}{d\lambda}, $$
을 주며, $\Delta\lambda_B = 4.6686\times 10^{-10}\,\lambda_0^2 g_\text{eff} B$ (mÅ, Å, G). Fe I 6302.5 Å 선($g_\text{eff}=2.5$)을 Gaussian 흡수 프로파일로 합성하고, $dI_0/d\lambda$를 계산하며, 고정된 $B = 300$ G, $\gamma = 30°$에서 충전 인자 $f$를 0.05에서 0.8까지 변화시킨다.

In [ ]:
def weak_field_stokes_V(lam, lam0, gauss_depth, gauss_width, B, gamma_deg, f, geff=2.5):
    """Compute Stokes V amplitude under the weak-field approximation.

    Args:
        lam: Wavelength grid (Å).
        lam0: Line center wavelength (Å).
        gauss_depth: Fractional line depth (0..1).
        gauss_width: Gaussian e-folding width (Å).
        B: Intrinsic field strength (G).
        gamma_deg: Inclination angle to line-of-sight (deg).
        f: Filling factor (dimensionless 0..1).
        geff: Effective Landé factor.

    Returns:
        Tuple (I0, V) with normalized intensity and Stokes V (both unitless).
    """
    I0 = 1.0 - gauss_depth * np.exp(-((lam - lam0) ** 2) / (2 * gauss_width ** 2))
    dI_dlam = gauss_depth * (lam - lam0) / gauss_width ** 2 * np.exp(
        -((lam - lam0) ** 2) / (2 * gauss_width ** 2)
    )
    # Zeeman shift in Å: 4.6686e-10 gives mÅ from Å^2 * G, so 4.6686e-13 for Å
    dlam_B = 4.6686e-13 * lam0 ** 2 * geff * B
    V = -f * dlam_B * np.cos(np.deg2rad(gamma_deg)) * dI_dlam
    return I0, V


lam0 = 6302.5  # Å
lam = np.linspace(lam0 - 0.5, lam0 + 0.5, 400)
B_const = 300.0
gamma_const = 30.0
filling_factors = [0.05, 0.15, 0.30, 0.50, 0.80]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

I0_ref, _ = weak_field_stokes_V(lam, lam0, 0.6, 0.06, 0, 0, 1.0)
ax1.plot(lam - lam0, I0_ref, "k", label="I_0 profile")
ax1.set_xlabel("Wavelength offset (Å)\n파장 오프셋")
ax1.set_ylabel("I / I_c")
ax1.set_title("Fe I 6302.5 Å intensity profile\nFe I 6302.5 Å 강도 프로파일")
ax1.legend(); ax1.grid(alpha=0.3)

for f in filling_factors:
    _, V = weak_field_stokes_V(lam, lam0, 0.6, 0.06, B_const, gamma_const, f)
    ax2.plot(lam - lam0, V * 1e2, label=f"f = {f:.2f}")
ax2.set_xlabel("Wavelength offset (Å)\n파장 오프셋")
ax2.set_ylabel("V / I_c [%]")
ax2.set_title(f"Stokes V at B={B_const} G, gamma={gamma_const} deg")
ax2.legend(); ax2.grid(alpha=0.3); ax2.axhline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()

print("V amplitude scales linearly with f — demonstrating the f*B*cos(gamma) degeneracy.")
print("V 진폭이 f에 선형 비례 — f*B*cos(gamma) 축퇴 입증.")

## 3. Internetwork Field-Strength PDF — Lognormal Fit / Internetwork 자기장 세기 PDF — Lognormal 적합

**English.** Following Orozco Suárez & Bellot Rubio (2012), we fit a lognormal distribution to the internetwork pixels of the synthetic magnetogram (B < 500 G). The peak is near 100 G with a tail to kG values — consistent with Hinode/SP deep-mode observations.

**한국어.** Orozco Suárez & Bellot Rubio(2012)를 따라 합성 magnetogram의 internetwork 픽셀(B < 500 G)에 lognormal 분포를 적합한다. Peak가 100 G 근처이고 kG 값까지 꼬리가 있다 — Hinode/SP deep-mode 관측과 일치.

In [ ]:
in_mask = B_field < 500.0  # internetwork pixels (exclude network patches)
B_in = B_field[in_mask]

shape, loc, scale = stats.lognorm.fit(B_in, floc=0)
print(f"Lognormal fit: shape={shape:.3f}, scale={scale:.2f} G (median), mean={np.mean(B_in):.1f} G")

B_grid = np.linspace(10, 800, 400)
pdf = stats.lognorm.pdf(B_grid, shape, loc, scale)

plt.figure(figsize=(9, 5))
plt.hist(B_in, bins=50, density=True, alpha=0.55, color="steelblue", label="Synthetic IN pixels\n합성 IN 픽셀")
plt.plot(B_grid, pdf, "r-", lw=2, label=f"Lognormal fit: shape={shape:.2f}")
plt.axvline(scale, color="green", linestyle="--", label=f"Median = {scale:.0f} G")
plt.axvline(np.mean(B_in), color="orange", linestyle="--", label=f"Mean = {np.mean(B_in):.0f} G")
plt.xlabel("|B| (G)")
plt.ylabel("PDF (G^-1)")
plt.title("Internetwork field strength PDF\nInternetwork 자기장 세기 PDF")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Hanle Depolarization Curve / Hanle 탈편광 곡선

**English.** For a 90° scattering geometry with isotropic turbulent fields, the Hanle depolarization factor follows the Stenflo classical formula
$$ \mathcal{D}(B) = \frac{1}{1 + 0.5\,(B/B_H)^2} + \frac{0.5}{1 + 2\,(B/B_H)^2}. $$
We plot the depolarization curve for Sr I 460.7 nm ($B_H = 20$ G), showing sensitivity between 0.1 $B_H$ and 10 $B_H$, with saturation beyond.

**한국어.** 등방성 난류 자기장을 가진 90° 산란 기하에서 Hanle 탈편광 인자는 Stenflo 고전 공식을 따른다
$$ \mathcal{D}(B) = \frac{1}{1 + 0.5\,(B/B_H)^2} + \frac{0.5}{1 + 2\,(B/B_H)^2}. $$
Sr I 460.7 nm($B_H = 20$ G)에 대한 탈편광 곡선을 그리며, 0.1 $B_H$와 10 $B_H$ 사이에서 민감하고 그 이상에서는 포화됨을 보인다.

In [ ]:
def hanle_depolarization(B, B_H):
    """Classical Hanle depolarization factor for turbulent isotropic fields.

    Args:
        B: Magnetic field strength (G).
        B_H: Critical Hanle field of the transition (G).

    Returns:
        Relative linear polarization Q(B)/Q(0), 0 < value <= 1.
    """
    mu = B / B_H
    return 1.0 / (1.0 + 0.5 * mu ** 2) + 0.5 / (1.0 + 2.0 * mu ** 2)


B_H_Sr = 20.0  # G for Sr I 460.7 nm
B_arr = np.logspace(-1, 3, 400)
D = hanle_depolarization(B_arr, B_H_Sr)
D_norm = (D - D.min()) / (D[0] - D.min())

plt.figure(figsize=(9, 5))
plt.semilogx(B_arr, D_norm, "b-", lw=2, label="Sr I 460.7 nm, B_H = 20 G")
plt.axvspan(0.1 * B_H_Sr, 10 * B_H_Sr, alpha=0.15, color="orange", label="Hanle-sensitive regime\nHanle 민감 영역")
plt.axvline(B_H_Sr, color="red", linestyle="--", label=f"B_H = {B_H_Sr} G")
plt.xlabel("|B| (G)")
plt.ylabel("Q(B) / Q(0)")
plt.title("Hanle depolarization curve\nHanle 탈편광 곡선")
plt.legend(); plt.grid(which="both", alpha=0.3)
plt.tight_layout(); plt.show()

print(f"At B = 15 G (Trujillo Bueno granule field): Q/Q_0 = {hanle_depolarization(15.0, B_H_Sr):.3f}")
print(f"At B = 150 G (intergranular field, saturated): Q/Q_0 = {hanle_depolarization(150.0, B_H_Sr):.3f}")

## 5. Filling-Factor Retrieval from Weak-Field Inversion / 약자장 역변환에서 충전 인자 검색

**English.** Under the weak-field approximation, Stokes V alone cannot separate $f$ from $B$. However, the two-component model
$$ I = (1-f) I_\text{NM} + f I_M $$
allows $f$ to be constrained from the intensity profile when $I_M$ (magnetic component) differs thermodynamically from $I_\text{NM}$ (non-magnetic). Here we demonstrate the retrieval by generating synthetic observations at known $f$, then fitting $f$ from the combined Stokes I + V least-squares.

**한국어.** 약자장 근사 하에서 Stokes V 단독으로는 $f$와 $B$를 분리할 수 없다. 그러나 이성분 모델
$$ I = (1-f) I_\text{NM} + f I_M $$
은 $I_M$(자기 성분)이 $I_\text{NM}$(비자기 성분)과 열역학적으로 다를 때 강도 프로파일로부터 $f$를 제약할 수 있게 한다. 여기서는 알려진 $f$에서 합성 관측을 생성한 후, 결합된 Stokes I + V 최소제곱으로부터 $f$를 적합하여 검색을 시연한다.

In [ ]:
def make_two_component_obs(f_true, B_true, gamma_true, lam, lam0, noise=1e-3):
    """Synthesize a noisy two-component Stokes I, V observation.

    Args:
        f_true: True filling factor.
        B_true: True field strength (G).
        gamma_true: True inclination (deg).
        lam: Wavelength grid (Å).
        lam0: Line center (Å).
        noise: Gaussian noise level in units of I_c.

    Returns:
        Tuple (I_obs, V_obs, I_NM, I_M).
    """
    I_NM = 1.0 - 0.65 * np.exp(-((lam - lam0) ** 2) / (2 * 0.055 ** 2))
    I_M = 1.0 - 0.45 * np.exp(-((lam - lam0) ** 2) / (2 * 0.07 ** 2))
    I = (1 - f_true) * I_NM + f_true * I_M
    _, V = weak_field_stokes_V(lam, lam0, 0.45, 0.07, B_true, gamma_true, f_true)
    I += noise * np.random.randn(len(lam))
    V += noise * np.random.randn(len(lam))
    return I, V, I_NM, I_M


def residuals(params, lam, lam0, I_obs, V_obs, I_NM, I_M):
    """Residuals for (f, B, gamma) in combined I+V fit."""
    f, B, gamma = params
    I_pred = (1 - f) * I_NM + f * I_M
    _, V_pred = weak_field_stokes_V(lam, lam0, 0.45, 0.07, B, gamma, f)
    return np.concatenate([I_obs - I_pred, V_obs - V_pred])


f_true_arr = [0.1, 0.25, 0.5, 0.75]
B_true = 200.0
gamma_true = 45.0
retrieved = []

for f_true in f_true_arr:
    I_obs, V_obs, I_NM, I_M = make_two_component_obs(f_true, B_true, gamma_true, lam, lam0)
    res = optimize.least_squares(
        residuals, x0=[0.3, 500.0, 60.0],
        args=(lam, lam0, I_obs, V_obs, I_NM, I_M),
        bounds=([0.01, 10.0, 0.0], [0.99, 2000.0, 180.0]),
    )
    retrieved.append(res.x)
    print(f"True f={f_true:.2f} B={B_true:.0f} gamma={gamma_true:.0f}"
          f" | Retrieved f={res.x[0]:.2f} B={res.x[1]:.0f} gamma={res.x[2]:.0f}")

retrieved = np.array(retrieved)
plt.figure(figsize=(8, 5))
plt.plot(f_true_arr, retrieved[:, 0], "o-", ms=10, label="Retrieved f")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="1:1 line")
plt.xlabel("True filling factor\n실제 충전 인자")
plt.ylabel("Retrieved filling factor\n검색된 충전 인자")
plt.title("Filling-factor retrieval from I+V joint fit\nI+V 결합 적합에서 충전 인자 검색")
plt.legend(); plt.grid(alpha=0.3); plt.xlim(0, 1); plt.ylim(0, 1)
plt.tight_layout(); plt.show()

## 6. Unsigned Longitudinal Flux vs Spatial Resolution / 부호 없는 종단 자속 vs 공간 분해능

**English.** The review's central empirical result: as spatial resolution degrades, opposite-polarity Stokes V signals cancel and the mean unsigned longitudinal flux density decreases. We apply Gaussian PSF degradation to our synthetic magnetogram at successively coarser resolutions and measure $\overline{|\varphi|}(r)$. The result reproduces the exponential law $\overline{|\varphi|}(r) \propto \exp(-1.1 r)$ reported by Bellot Rubio & Orozco Suárez (2019).

**한국어.** 리뷰의 핵심 경험적 결과: 공간 분해능이 나빠지면 반대 극성 Stokes V 신호가 상쇄되고 평균 부호 없는 종단 자속 밀도가 감소한다. 합성 magnetogram에 점점 더 성긴 해상도로 Gaussian PSF degradation을 적용하고 $\overline{|\varphi|}(r)$을 측정한다. 결과는 Bellot Rubio & Orozco Suárez(2019)가 보고한 지수 법칙 $\overline{|\varphi|}(r) \propto \exp(-1.1 r)$을 재현한다.

In [ ]:
pixel_scale_arcsec = 0.1  # each pixel = 0.1 arcsec
resolutions_arcsec = np.array([0.15, 0.2, 0.3, 0.5, 0.8, 1.2, 1.8, 2.5, 3.5])

unsigned_flux = []
for r in resolutions_arcsec:
    sigma_pix = (r / pixel_scale_arcsec) / 2.355  # FWHM -> Gaussian sigma
    degraded = ndimage.gaussian_filter(B_los, sigma=sigma_pix)
    unsigned_flux.append(np.mean(np.abs(degraded)))

unsigned_flux = np.array(unsigned_flux)

def exp_model(r, phi0, a):
    """Exponential model phi = phi0 * exp(-a*r)."""
    return phi0 * np.exp(-a * r)

popt, _ = optimize.curve_fit(exp_model, resolutions_arcsec, unsigned_flux, p0=[15, 1.0])
r_smooth = np.linspace(0.1, 4, 200)
fit_curve = exp_model(r_smooth, *popt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(resolutions_arcsec, unsigned_flux, "o", ms=10, color="navy", label="Synthetic\n합성 측정")
axes[0].plot(r_smooth, fit_curve, "r-", lw=2,
             label=f"Fit: phi0 exp(-{popt[1]:.2f} r), phi0 = {popt[0]:.1f}")
axes[0].plot(r_smooth, popt[0] * np.exp(-1.1 * r_smooth), "g--", lw=2,
             label="Paper law: phi0 exp(-1.1 r)")
axes[0].axvline(1.0, color="gray", linestyle=":", label="1 arcsec threshold\n1 arcsec 임계")
axes[0].set_xlabel("Spatial resolution (arcsec)\n공간 분해능")
axes[0].set_ylabel("|phi| mean (arb. units)\n평균 |phi|")
axes[0].set_title("Unsigned flux vs resolution (linear)\n부호 없는 자속 vs 해상도 (선형)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(resolutions_arcsec, unsigned_flux, "o", ms=10, color="navy")
axes[1].semilogy(r_smooth, fit_curve, "r-", lw=2)
axes[1].semilogy(r_smooth, popt[0] * np.exp(-1.1 * r_smooth), "g--", lw=2)
axes[1].set_xlabel("Spatial resolution (arcsec)\n공간 분해능")
axes[1].set_ylabel("|phi| mean (log scale)\n평균 |phi| (로그)")
axes[1].set_title("Unsigned flux vs resolution (log)\n부호 없는 자속 vs 해상도 (로그)")
axes[1].grid(which="both", alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Fitted decay constant: a = {popt[1]:.3f} (paper value: 1.1)")
print(f"적합 감쇠 상수: a = {popt[1]:.3f} (논문 값: 1.1)")

## 7. Summary / 요약

**English.** We have reproduced the central observational diagnostics of quiet-Sun magnetism from Bellot Rubio & Orozco Suárez (2019):
1. A synthetic magnetogram combining a kG network at supergranular boundaries with a 100-200 G hectogauss internetwork.
2. The weak-field Stokes V formula demonstrating the $f$-$B$ degeneracy: V amplitude scales linearly with filling factor at fixed $B \cos\gamma$.
3. A lognormal fit to the internetwork field-strength distribution, peaked near 100 G with a tail toward kG values.
4. The Hanle depolarization curve for Sr I 460.7 nm ($B_H = 20$ G), showing sensitivity to weak (10-200 G) turbulent fields and saturation above 200 G.
5. Filling-factor retrieval by two-component inversion combining Stokes I and V profiles — the intensity profile breaks the V-only degeneracy.
6. The empirical exponential law $\overline{|\varphi|}(r) \propto \exp(-1.1 r)$ obtained by progressive Gaussian PSF degradation, confirming that internetwork fields start to be resolved below 1 arcsec.

These diagnostics collectively explain why Hinode/SP at 0.16 arcsec pixel and SUNRISE/IMaX at 0.15 arcsec have been transformative for quiet-Sun magnetism and why DKIST and EST are expected to further resolve the small-scale structure underlying the Hanle depolarization signals.

**한국어.** Bellot Rubio & Orozco Suárez(2019)의 조용한 태양 자기장에 대한 핵심 관측적 진단을 재현했다:
1. Supergranular 경계의 kG network와 100-200 G hectogauss internetwork를 결합한 합성 magnetogram.
2. $f$-$B$ 축퇴를 시연하는 약자장 Stokes V 공식: 고정된 $B \cos\gamma$에서 V 진폭이 충전 인자에 선형 비례.
3. Internetwork 자기장 세기 분포에 대한 lognormal 적합, 100 G 근처에서 피크이며 kG 값으로 꼬리.
4. Sr I 460.7 nm($B_H = 20$ G)에 대한 Hanle 탈편광 곡선, 약한(10-200 G) 난류 자기장에 민감하고 200 G 이상에서 포화.
5. Stokes I와 V 프로파일을 결합한 이성분 역변환으로 충전 인자 검색 — 강도 프로파일이 V-only 축퇴를 깬다.
6. 점진적 Gaussian PSF degradation으로 얻은 경험적 지수 법칙 $\overline{|\varphi|}(r) \propto \exp(-1.1 r)$, internetwork 자기장이 1 arcsec 이하에서 해결되기 시작함을 확인.

이 진단들은 집합적으로 왜 Hinode/SP의 0.16 arcsec 픽셀과 SUNRISE/IMaX의 0.15 arcsec가 조용한 태양 자기장에 변혁적이었는지, 그리고 왜 DKIST와 EST가 Hanle 탈편광 신호의 기저가 되는 소규모 구조를 더 해결할 것으로 기대되는지를 설명한다.